### 1. Resumen extractivo (basado en frecuencia)

In [3]:
import heapq
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

nltk.download('punkt')
nltk.download('stopwords')

def resumen_extractivo(texto, num_frases=2):
    # 1. Preprocesamiento: limpieza y frecuencias
    stop_words = set(stopwords.words('spanish'))
    palabras = word_tokenize(texto.lower())

    frecuencia_palabras = {}
    for palabra in palabras:
        if palabra.isalpha() and palabra not in stop_words:
            frecuencia_palabras[palabra] = frecuencia_palabras.get(palabra, 0) + 1

    # Normalizar frecuencias (0 a 1)
    max_frec = max(frecuencia_palabras.values())
    for palabra in frecuencia_palabras:
        frecuencia_palabras[palabra] /= max_frec

    # 2. Puntuar frases basándose en las palabras que contienen
    frases = sent_tokenize(texto)
    puntuacion_frases = {}
    for frase in frases:
        for palabra in word_tokenize(frase.lower()):
            if palabra in frecuencia_palabras:
                puntuacion_frases[frase] = puntuacion_frases.get(frase, 0) + frecuencia_palabras[palabra]

    # 3. Seleccionar las mejores frases
    resumen = heapq.nlargest(num_frases, puntuacion_frases, key=puntuacion_frases.get)
    return ' '.join(resumen)

texto_ejemplo = """El aprendizaje profundo es una rama de la
inteligencia artificial.
Utiliza redes neuronales complejas para procesar datos.
Estas redes imitan el comportamiento del cerebro humano.
Es fundamental para tecnologías como el reconocimiento de voz y la
visión por computadora."""

print(resumen_extractivo(texto_ejemplo))

Utiliza redes neuronales complejas para procesar datos. Estas redes imitan el comportamiento del cerebro humano.


[nltk_data] Downloading package punkt to /home/ciabd10/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/ciabd10/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### 2. Resumen abstractivo (deep learning)

In [10]:
from transformers import pipeline

# TinyLlama es pequeño, moderno y entiende mucho mejor las instrucciones
generator = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", device=-1)

texto = "La inteligencia artificial busca crear sistemas que imiten la inteligencia humana para tareas de aprendizaje y razonamiento."

# Usamos un formato de instrucción que el modelo entiende mejor
prompt = f"<|system|>\nResume el siguiente texto en una frase corta.</s>\n<|user|>\n{texto}</s>\n<|assistant|>\n"

# Generamos el resultado
# max_new_tokens controla cuánto escribe el modelo desde cero
resultado = generator(prompt, max_new_tokens=50, do_sample=True, temperature=0.7)

# Limpiamos la respuesta para ver solo lo que dijo el asistente
print("-" * 30)
print("RESUMEN GENERADO:")
respuesta = resultado[0]['generated_text'].split("<|assistant|>\n")[-1]
print(respuesta.strip())
print("-" * 30)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11882.89it/s]
Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


------------------------------
RESUMEN GENERADO:
La inteligencia artificial (IA) busca crear sistemas artificiales que imitan la inteligencia humana para el proceso de aprendizaje y la razonamiento. Estas son las tres grandes áreas de investig
------------------------------


### 3. Extracción de ideas clave (keywords)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def ideas_clave(texto, n=5):
    # Usamos TF-IDF para encontrar palabras "importantes"
    vectorizer = TfidfVectorizer(stop_words=stopwords.words('spanish'))
    tfidf_matrix = vectorizer.fit_transform([texto])

    # Obtenemos los nombres de las palabras y su puntuación
    palabras = vectorizer.get_feature_names_out()
    puntuaciones = tfidf_matrix.toarray()[0]

    # Ordenamos y sacamos las n mejores
    indices_top = puntuaciones.argsort()[-n:][::-1]
    return [palabras[i] for i in indices_top]

print(f"Ideas clave: {ideas_clave(texto_ejemplo)}")

Ideas clave: ['redes', 'voz', 'visión', 'utiliza', 'tecnologías']
